# Imports

In [ ]:
from glob import glob

import dask.dataframe as dd
import lsdb
import pandas as pd
from dask.distributed import Client, LocalCluster

from science_catalogs import open_lsdb_catalog, prepare_catalog, write_catalog

# Paths

In [ ]:
path_to_input = "../demo_catalog/data/input/*.parquet"
path_to_config = "../demo_catalog/configs/demo_hats.yml"
path_to_output_catalog = "./output/demo_hats_catalog"
output_format = "hats"

# path_to_input = "../demo_catalog/data/input/*.parquet"
# path_to_config = "../demo_catalog/configs/demo_parquet.yml"
# path_to_output_catalog = "./output/*.parquet"
# output_format = "parquet"

# Inspect input catalog

In [ ]:
def dask_local_compute(func, *args, **kwargs):
    with LocalCluster(
        n_workers=3,
        threads_per_worker=2,
        memory_limit="4GB",
        dashboard_address=None,
    ) as cluster:
        with Client(cluster):
            return func(*args, **kwargs)

In [ ]:
ddf = dd.read_parquet(path_to_input)

In [ ]:
head = dask_local_compute(lambda: ddf.head(4))
head

# Prepare a catalog from a catalog-processing YAML configuration

In [ ]:
prepared = prepare_catalog(path_to_config)

In [ ]:
type(prepared)

In [ ]:
prepared

# Write the result to disk

In [ ]:
written_paths = write_catalog(prepared, "./output")

# Open the prepared catalog with LSDB

In [ ]:
if output_format == "hats":
    catalog = open_lsdb_catalog(path_to_output_catalog)
elif output_format == "parquet":
    df = pd.read_parquet(glob(path_to_output_catalog))
    catalog = lsdb.from_dataframe(df)

In [ ]:
type(catalog)

In [ ]:
catalog

# Inspect input catalog

In [ ]:
head = dask_local_compute(lambda: catalog.head(4))
head